# Rho Gradient Analysis in AD EEG
## Port-Hamiltonian Prediction 1

Dataset: OpenNeuro ds004504 (36 AD, 23 FTD, 29 controls)

In [ ]:
!pip install -q mne openneuro-py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from scipy import signal
from scipy.stats import ttest_ind, spearmanr
import warnings, os
from pathlib import Path
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

## 1. Download

In [ ]:
import openneuro
DL='/content/eeg'
os.makedirs(DL,exist_ok=True)
openneuro.download(dataset='ds004504',target_dir=DL,include=['participants.tsv','derivatives/'])

## 2. Metadata

In [ ]:
pf=list(Path(DL).rglob('participants.tsv'))
pts=pd.read_csv(pf[0],sep='\t') if pf else pd.DataFrame()
print(pts['Group'].value_counts())

## 3. Functions

In [ ]:
CG={'dorsal':['C3','Cz','C4'],'ventral_temporal':['T5','T6'],
    'lateral_temporal':['T3','T4'],'frontal':['F3','Fz','F4'],
    'frontopolar':['Fp1','Fp2'],'parietal':['P3','Pz','P4'],'occipital':['O1','O2']}
CA={'T7':'T3','T8':'T4','P7':'T5','P8':'T6'}
DV={'dorsal':1.0,'parietal':0.7,'frontal':0.6,'frontopolar':0.4,
    'lateral_temporal':0.2,'occipital':0.3,'ventral_temporal':0.0}
FB={'delta':(1,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'broadband':(1,45)}

def rho_ar2(s):
    s=(s-s.mean())/(s.std()+1e-12)
    Y,X=s[2:],np.column_stack([s[1:-1],s[:-2]])
    c=np.linalg.lstsq(X,Y,rcond=None)[0]
    return np.max(np.abs(np.roots([1,-c[0],-c[1]])))

def rho_ep(s,fs,ep=4.0):
    n=int(ep*fs);st=n//2;rs=[]
    for i in range(0,len(s)-n,st):
        seg=s[i:i+n]
        if seg.std()<1e-10:continue
        r=rho_ar2(seg)
        if 0<r<1:rs.append(r)
    return np.median(rs) if len(rs)>=3 else np.nan

def bpf(s,fs,lo,hi):
    b,a=signal.butter(4,[lo/(fs/2),hi/(fs/2)],btype='band')
    return signal.filtfilt(b,a,s)

## 4. Process subjects

In [ ]:
files=[]
for e in ['*.set','*.fif','*.edf']:
    for f in sorted(Path(DL).rglob(e)):
        sub=next((p for p in f.parts if p.startswith('sub-')),None)
        if sub:files.append((sub,str(f)))
print(f'{len(files)} files found')

In [ ]:
res=[]
for i,(sid,fp) in enumerate(files):
    print(f'{sid} ({i+1}/{len(files)})',end=' ')
    try:
        if fp.endswith('.set'):raw=mne.io.read_raw_eeglab(fp,preload=True)
        elif fp.endswith('.edf'):raw=mne.io.read_raw_edf(fp,preload=True)
        else:raw=mne.io.read_raw_fif(fp,preload=True)
        rm={ch:CA[ch.upper()] for ch in raw.ch_names if ch.upper() in CA}
        if rm:raw.rename_channels(rm)
        raw.filter(0.5,45,verbose=False);raw.notch_filter(50,verbose=False)
        sn=sid.replace('sub-','')
        row=pts[pts['participant_id'].str.contains(sn)]
        g=row['Group'].values[0] if len(row) else '?'
        mm=row['MMSE'].values[0] if len(row) else np.nan
        d={'subject':sid,'group':g,'mmse':mm}
        fs=raw.info['sfreq']
        for bn,(lo,hi) in FB.items():
            for gn,chs in CG.items():
                vs=[]
                for ch in chs:
                    if ch in raw.ch_names:
                        x=raw.get_data(picks=[ch])[0]
                        if bn!='broadband':x=bpf(x,fs,lo,hi)
                        r=rho_ep(x,fs)
                        if not np.isnan(r):vs.append(r)
                d[f'rho_{bn}_{gn}']=np.mean(vs) if vs else np.nan
        res.append(d);print(f'OK({g})')
    except Exception as e:print(f'ERR:{e}')
df=pd.DataFrame(res)
print(f'Done: {len(df)} subjects')

## 5. Results

In [ ]:
for b in FB:
    dk,vk=f'rho_{b}_dorsal',f'rho_{b}_ventral_temporal'
    if dk in df and vk in df:df[f'dv_{b}']=df[dk]-df[vk]

GL={'A':'AD','F':'FTD','C':'Control'}
GC={'AD':'#d62728','FTD':'#ff7f0e','Control':'#2e75b6'}

fig,ax=plt.subplots(1,3,figsize=(16,5))
# Gradient by DV position
for grp in ['C','A','F']:
    g=df[df['group']==grp]
    ms,es,ps=[],[],[]
    for reg,dv in sorted(DV.items(),key=lambda x:x[1]):
        k=f'rho_broadband_{reg}'
        if k in g:v=g[k].dropna()
        if len(v):ms.append(v.mean());es.append(v.sem());ps.append(dv)
    if ms:ax[0].errorbar(ps,ms,yerr=es,marker='o',lw=2,capsize=4,label=GL.get(grp,grp),color=GC.get(GL.get(grp,''),'gray'))
ax[0].set_xlabel('DV pos');ax[0].set_ylabel('rho');ax[0].legend();ax[0].set_title('Broadband rho gradient')

# Boxplot
if 'dv_broadband' in df:
    gd=[{'G':GL.get(g,g),'V':v} for g in ['C','A','F'] for v in df[df['group']==g]['dv_broadband'].dropna()]
    sns.boxplot(data=pd.DataFrame(gd),x='G',y='V',ax=ax[1],order=['Control','AD','FTD'],palette=[GC[g] for g in ['Control','AD','FTD']])
    ax[1].axhline(0,color='k',ls='--',alpha=0.3);ax[1].set_title('DV gradient');ax[1].set_ylabel('rho(D)-rho(V)')

# MMSE
if 'dv_broadband' in df:
    m=df['mmse'].notna()&df['dv_broadband'].notna()
    if m.sum()>10:
        for grp in df['group'].unique():
            gm=m&(df['group']==grp);lab=GL.get(grp,grp)
            ax[2].scatter(df.loc[gm,'mmse'],df.loc[gm,'dv_broadband'],c=GC.get(lab,'gray'),s=50,alpha=0.7,label=lab)
        r,p=spearmanr(df.loc[m,'mmse'],df.loc[m,'dv_broadband'])
        ax[2].set_xlabel('MMSE');ax[2].set_ylabel('DV gradient');ax[2].set_title(f'r={r:.3f}, p={p:.4f}');ax[2].legend()
plt.tight_layout();plt.savefig('/content/result.png',dpi=150);plt.show()

## 6. Stats

In [ ]:
for b in FB:
    gk=f'dv_{b}'
    if gk not in df:continue
    print(f'\n--- {b.upper()} ---')
    ad,cn=df[df['group']=='A'][gk].dropna(),df[df['group']=='C'][gk].dropna()
    if len(ad)>3 and len(cn)>3:
        t,p=ttest_ind(ad,cn)
        d=(ad.mean()-cn.mean())/np.sqrt((ad.var()+cn.var())/2)
        print(f'AD: {ad.mean():.5f}+/-{ad.sem():.5f}  CN: {cn.mean():.5f}+/-{cn.sem():.5f}')
        print(f't={t:.3f}, p={p:.4f}, d={d:.3f}')

In [ ]:
df.to_csv('/content/rho_results.csv',index=False)
print('Saved!')